In [1]:
import argparse
import os
from typing import Tuple
import numpy as np
from scipy.special import lpmv
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import uniform_filter1d

In [2]:
def build_grid(L: float, N: int) -> Tuple[np.ndarray, float]:
    """Build spatial grid for the system."""
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    return x, dx


def build_bound_states(lam: int, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Construct bound states for the Pöschl–Teller potential."""
    u = np.tanh(x)
    mus = np.arange(lam, 0, -1)
    states = len(mus)
    raw = np.zeros((states, x.size), dtype=float)
    for i, mu in enumerate(mus):
        raw[i, :] = lpmv(int(mu), int(lam), u)
    return mus, raw


def normalize_psis(raw_psis: np.ndarray, x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Normalize the wavefunctions."""
    states = raw_psis.shape[0]
    psis = np.zeros_like(raw_psis)
    norms = np.zeros(states)
    for i in range(states):
        sq = np.abs(raw_psis[i, :] ** 2)
        norm = np.trapz(sq, x)
        norms[i] = norm
        psis[i, :] = raw_psis[i, :] / np.sqrt(norm)
    return psis, norms


def compute_matrices(psis: np.ndarray, x: np.ndarray, dx: float) -> Tuple[np.ndarray, np.ndarray]:
    """Compute x and p operator matrices."""
    n = psis.shape[0]
    x_mat = np.zeros((n, n), dtype=float)
    p_mat = np.zeros((n, n), dtype=complex)

    dpsis = np.gradient(psis, dx, axis=1)

    for i in range(n):
        for j in range(n):
            x_mat[i, j] = np.trapz(psis[i, :] * x * psis[j, :], x)
            p_mat[i, j] = (-1j) * np.trapz(psis[i, :] * dpsis[j, :], x)

    x_mat = np.real_if_close(x_mat, tol=1e6)
    return x_mat, p_mat


def energies_from_mu(mu: np.ndarray) -> np.ndarray:
    """Compute energy eigenvalues from mu."""
    return -0.5 * (mu.astype(float) ** 2)

In [3]:
def microcanonical_OTOC(Es: np.ndarray, x_mat: np.ndarray, p_mat: np.ndarray,
                        mu_index: int, t_array: np.ndarray) -> np.ndarray:
    """Compute microcanonical OTOC for a given state."""
    n = len(Es)
    Cvals = np.zeros_like(t_array, dtype=float)
    deltaE = Es[:, None] - Es[None, :]

    for k, t in enumerate(t_array):
        phases = np.exp(1j * deltaE * t)
        x_t = x_mat * phases
        comm = x_t.dot(p_mat) - p_mat.dot(x_t)
        comm2 = comm.dot(comm)
        Cvals[k] = -np.real(comm2[mu_index, mu_index])

    return Cvals


def thermal_OTOC(Es: np.ndarray, x_mat: np.ndarray, p_mat: np.ndarray,
                 beta: float, t_array: np.ndarray) -> np.ndarray:
    """Compute thermal OTOC."""
    weights = np.exp(-beta * Es)
    Z = np.sum(weights)
    Csum = np.zeros_like(t_array, dtype=float)

    for mu_index in range(len(Es)):
        Cmu = microcanonical_OTOC(Es, x_mat, p_mat, mu_index, t_array)
        Csum += weights[mu_index] * Cmu

    return Csum / Z

In [4]:
def save_plot(fig, outpath: str, dpi: int = 150):
    """Save matplotlib figure."""
    fig.savefig(outpath, dpi=dpi, bbox_inches="tight")


def adaptive_downsample(x, y, max_points=5000):
    """Downsample the data for clearer visualization if too dense."""
    if len(x) > max_points:
        idx = np.linspace(0, len(x) - 1, max_points).astype(int)
        return x[idx], y[idx]
    return x, y


def smooth_data(y, smooth_window=5):
    """Apply mild smoothing for dense oscillations."""
    if len(y) > smooth_window * 2:
        return uniform_filter1d(y, size=smooth_window)
    return y


def plot_time_series(t, C, ylabel="", title="", outpath=None, dpi=150, alpha=None):
    """Plot time-domain OTOC with adaptive handling."""
    alpha = alpha or (0.7 if len(t) < 2000 else 0.4)
    t_plot, C_plot = adaptive_downsample(t, smooth_data(C))

    plt.figure(figsize=(8, 5))
    plt.plot(t_plot, C_plot, linewidth=1.2, alpha=alpha)
    plt.xlabel("t")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    if outpath:
        plt.savefig(outpath, dpi=dpi)
        plt.close()
    else:
        plt.show()


def plot_fft(t, C, outpath=None, dpi=150):
    """Plot FFT spectrum of OTOC."""
    freqs = np.fft.fftfreq(len(t), d=t[1] - t[0])
    fft_vals = np.abs(np.fft.fft(C))
    freqs, fft_vals = adaptive_downsample(freqs, fft_vals)

    plt.figure(figsize=(8, 5))
    plt.plot(freqs, fft_vals, linewidth=1.2, alpha=0.6)
    plt.xlabel("Frequency")
    plt.ylabel("|FFT(C)|")
    plt.title("FFT Spectrum")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    if outpath:
        plt.savefig(outpath, dpi=dpi)
        plt.close()
    else:
        plt.show()

In [5]:
def run_for_lambda(lam: int, args):
    """Run full OTOC computation for a given lambda."""
    outdir = os.path.join(args.outdir, f"lambda_{lam}")
    os.makedirs(outdir, exist_ok=True)

    x, dx = build_grid(args.L, args.N)
    mus, raw = build_bound_states(lam, x)
    psis, norms = normalize_psis(raw, x)
    x_mat, p_mat = compute_matrices(psis, x, dx)
    Es = energies_from_mu(mus)

    t = np.linspace(0, args.tmax, args.nt)

    mid_idx = len(mus) // 2
    C_ground = microcanonical_OTOC(Es, x_mat, p_mat, 0, t)
    C_mid = microcanonical_OTOC(Es, x_mat, p_mat, mid_idx, t)

    C_temps = {beta: thermal_OTOC(Es, x_mat, p_mat, beta, t) for beta in args.betas}

    # Save CSV summary
    df = pd.DataFrame({"mu": mus, "E_mu": Es, "norm_raw": norms})
    df.to_csv(os.path.join(outdir, "bound_states_summary.csv"), index=False)

    # Generate plots
    if args.save:
        plot_time_series(
            t, C_ground, ylabel="C_ground(t)",
            title=f"Microcanonical OTOC (ground), lambda={lam}",
            outpath=os.path.join(outdir, "C_ground.png"), dpi=args.dpi
        )
        plot_time_series(
            t, C_mid, ylabel="C_mid(t)",
            title=f"Microcanonical OTOC (mu={mus[mid_idx]}), lambda={lam}",
            outpath=os.path.join(outdir, "C_mid.png"), dpi=args.dpi
        )
        for beta, C in C_temps.items():
            plot_time_series(
                t, C, ylabel="C_T(t)",
                title=f"Thermal OTOC (beta={beta}), lambda={lam}",
                outpath=os.path.join(outdir, f"C_T_beta_{beta}.png"), dpi=args.dpi
            )

        if args.fft:
            plot_fft(t, C_ground, outpath=os.path.join(outdir, "fft_ground.png"), dpi=args.dpi)
            for beta, C in C_temps.items():
                plot_fft(t, C, outpath=os.path.join(outdir, f"fft_beta_{beta}.png"), dpi=args.dpi)
    else:
        plot_time_series(t, C_ground, ylabel="C_ground(t)",
                         title=f"Microcanonical OTOC (ground), lambda={lam}")
        plot_time_series(t, C_mid, ylabel=f"C_mid(t)",
                         title=f"Microcanonical OTOC (mu={mus[mid_idx]}), lambda={lam}")
        for beta, C in C_temps.items():
            plot_time_series(t, C, ylabel="C_T(t)",
                             title=f"Thermal OTOC (beta={beta}), lambda={lam}")
        if args.fft:
            plot_fft(t, C_ground)
            for beta, C in C_temps.items():
                plot_fft(t, C)

    print(f"Results written to {outdir}")
    return outdir

In [8]:
def parse_args():
    parser = argparse.ArgumentParser(description="Compute OTOCs for Pöschl-Teller potential")
    parser.add_argument("--lams", type=int, nargs="+", default=[5], help="list of integer lambda values")
    parser.add_argument("--L", type=float, default=12.0, help="half-width of spatial box [-L,L]")
    parser.add_argument("--N", type=int, default=20001, help="number of spatial grid points")
    parser.add_argument("--tmax", type=float, default=200.0, help="max time")
    parser.add_argument("--nt", type=int, default=2001, help="number of time samples")
    parser.add_argument("--betas", type=float, nargs="+", default=[10.0, 1.0, 0.1],
                        help="list of beta values for thermal OTOC")
    parser.add_argument("--save", action="store_true", help="save plots and data to disk instead of showing")
    parser.add_argument("--outdir", type=str, default="/mnt/data/pt_otoc_results", help="output directory")
    parser.add_argument("--dpi", type=int, default=150, help="dpi for saved figures")
    parser.add_argument("--fft", action="store_true", help="compute and save/plot FFT spectra")
    return parser.parse_args()

In [10]:
import traceback

if __name__ == "__main__":
    args = parse_args([])
    print("Starting Pöschl–Teller OTOC computations")

    for lam in args.lams:
        try:
            print(f"\n>>> Running for λ = {lam}")
            run_for_lambda(lam, args)
        except Exception as e:
            print(f"\n[ERROR] Exception occurred while running λ = {lam}")
            print("Error message:", e)
            print("\nFull traceback:")
            traceback.print_exc()
            print("-" * 80)

    print("All done.")


TypeError: parse_args() takes 0 positional arguments but 1 was given